# Gate features

One row per slate (not per candidate). They feed the gate `w = sigmoid(a·log1p(history_length) + b·log1p(time_since_last_activity) + c·log1p(purchase_count) + bias)` that mixes Model 1 (short-term) and Model 2 (long-term): `score = w·P1 + (1-w)·P2`. For very short histories the gate is overridden to `w = 1` (Model 1 only).

**Inputs:** `data/train_sampled.parquet`, `val.parquet`, `test.parquet` (only the history columns are read).
**Outputs:** `features/gate_train.parquet`, `gate_val.parquet`, `gate_test.parquet`.

Same leakage rule as the Model 1 features: history events younger than MIN_AGE_DAYS are dropped only if the item is one of the slate’s own candidates (FILTER_MODE = "in_slate"). Those are the slate’s own click/cart/order events. Recent events on other products are real session activity and are kept.

## 0. Setup

In [7]:
import os
import numpy as np, pandas as pd, pyarrow.parquet as pq

DATA = "data"
OUT = "features"
FILTER_MODE = "in_slate"   # must match the Model 1 notebook
MIN_AGE_DAYS = 0.05        # must match the Model 1 notebook
SHORT_HISTORY_PCTL = 10    # hard override w=1 below this train percentile of history length
os.makedirs(OUT, exist_ok=True)

# candidate_wid_list is only needed to build the in-slate keys
HIST = ["slate_id", "candidate_wid_list", "history_wid_list", "history_type_list", "history_days_before"]


def load(path, columns=None):
    return pq.read_table(path, columns=columns).to_pandas(ignore_metadata=True)


def slate_keys(df):
    """One int64 key per (slate, candidate): slate_id * 2**32 + wid."""
    lens = df["candidate_wid_list"].map(len).to_numpy()
    sid = np.repeat(df["slate_id"].to_numpy().astype(np.int64), lens)
    wid = np.concatenate(df["candidate_wid_list"].to_numpy()).astype(np.int64)
    return sid * (1 << 32) + wid


def clean_history(h, keys, mode=None, min_age=None):
    mode = FILTER_MODE if mode is None else mode
    min_age = MIN_AGE_DAYS if min_age is None else min_age
    if mode == "none":
        return h
    recent = h["days"].to_numpy() < min_age
    if mode == "all":
        return h[~recent]
    if mode == "in_slate":
        hk = h["slate_id"].to_numpy().astype(np.int64) * (1 << 32) + h["wid"].to_numpy()
        return h[~(recent & np.isin(hk, keys))]
    raise ValueError("mode must be 'in_slate', 'all' or 'none'")

## 1. Gate feature definitions
- `gate_history_length`: number of history events (all types) left after the in-slate filter.
- `gate_time_since_last_activity`: days since the most recent such event. Empty history -> filled with the maximum observed age (365) and flagged.
- `gate_purchase_count`: number of ORD events in history. Orders are observed over the full year, so unlike clicks and carts this count is comparable across users.
- `log1p` versions are what enter the sigmoid (long right tails).

In [8]:
def build_gate_features(df, keys=None, mode=None, min_age=None):
    n_rows = len(df)
    lens = df["history_wid_list"].map(len).to_numpy()
    h = pd.DataFrame({
        "slate_id": np.repeat(df["slate_id"].to_numpy(), lens),
        "row": np.repeat(np.arange(n_rows), lens),
        "wid": np.concatenate(df["history_wid_list"].to_numpy()).astype(np.int64),
        "typ": np.concatenate(df["history_type_list"].to_numpy()).astype(str),
        "days": np.concatenate(df["history_days_before"].to_numpy()).astype(np.float32),  # float32, same as Model 1
    })
    h = clean_history(h, slate_keys(df) if keys is None else keys, mode, min_age)
    row, typ, days = h["row"].to_numpy(), h["typ"].to_numpy(), h["days"].to_numpy().astype(np.float64)

    length = np.bincount(row, minlength=n_rows)
    purchases = np.bincount(row[typ == "ORD"], minlength=n_rows)
    last = pd.Series(days).groupby(row).min().reindex(range(n_rows))
    empty = length == 0
    last = last.fillna(365.0).to_numpy()

    out = pd.DataFrame({
        "slate_id": df["slate_id"].to_numpy(),
        "gate_history_length": length.astype(np.int32),
        "gate_time_since_last_activity": last.astype(np.float32),
        "gate_purchase_count": purchases.astype(np.int32),
        "gate_empty_history": empty.astype(np.int8),
    })
    out["gate_log_history_length"] = np.log1p(out["gate_history_length"]).astype(np.float32)
    out["gate_log_time_since_last"] = np.log1p(out["gate_time_since_last_activity"]).astype(np.float32)
    out["gate_log_purchase_count"] = np.log1p(out["gate_purchase_count"]).astype(np.float32)
    return out

## 2. Build for all splits

In [17]:
gate = {}
for name, fn in [("train", "train_sampled"), ("val", "val"), ("test", "test")]:
    d = load(f"{DATA}/{fn}.parquet", columns=HIST)
    gate[name] = build_gate_features(d)
    print(name, gate[name].shape)
#    del d

train (139882, 8)
val (16924, 8)
test (17025, 8)


In [18]:
import os, time
print("working dir:", os.getcwd())
print("output folder:", os.path.abspath(OUT))
for f in sorted(os.listdir(OUT)):
    p = os.path.join(OUT, f)
    print(f, f"{os.path.getsize(p)/1e6:.1f} MB", time.ctime(os.path.getmtime(p)))

working dir: /Users/sunxinyang/Documents/bt4222-grp8
output folder: /Users/sunxinyang/Documents/bt4222-grp8/features
ablation 0.0 MB Wed Oct 28 17:55:46 2026
ablation_tfidf 0.0 MB Wed Oct 28 18:05:59 2026
features_test.parquet 25.6 MB Wed Oct 28 17:27:42 2026
features_train.parquet 56.1 MB Wed Oct 28 17:26:52 2026
features_val.parquet 25.7 MB Wed Oct 28 17:27:16 2026
gate_test.parquet 0.4 MB Sun Oct 25 20:32:16 2026
gate_train.parquet 3.0 MB Sun Oct 25 20:32:16 2026
gate_val.parquet 0.4 MB Sun Oct 25 20:32:16 2026


## 3. Short-history override threshold
The threshold is the 10th percentile of `gate_history_length` on **train only**, recomputed after the recent-event filter (the earlier value of 10 was computed before the filter).

In [19]:
thr = int(np.percentile(gate["train"]["gate_history_length"], SHORT_HISTORY_PCTL))
print("short-history threshold (train 10th percentile):", thr)
for name, g in gate.items():
    g["gate_short_history"] = (g["gate_history_length"] <= thr).astype(np.int8)
    g.to_parquet(f"{OUT}/gate_{name}.parquet", index=False)
    print(name, "override share:", round(g["gate_short_history"].mean() * 100, 2), "% | empty histories:", int(g["gate_empty_history"].sum()))

short-history threshold (train 10th percentile): 10
train override share: 10.6 % | empty histories: 139
val override share: 10.61 % | empty histories: 33
test override share: 10.55 % | empty histories: 32


## 4. Descriptive statistics

In [20]:
cols = ["gate_history_length", "gate_time_since_last_activity", "gate_purchase_count"]
for name, g in gate.items():
    print("=====", name)
    print(g[cols].describe(percentiles=[.1, .25, .5, .75, .9, .99]).round(3).to_string())

===== train
       gate_history_length  gate_time_since_last_activity  gate_purchase_count
count           139882.000                     139882.000           139882.000
mean               152.706                          6.084               39.071
std                224.001                         25.290               70.081
min                  0.000                          0.000                0.000
10%                 10.000                          0.000                1.000
25%                 28.000                          0.001                4.000
50%                 78.000                          0.332               15.000
75%                187.000                          2.361               44.000
90%                375.000                         10.714              103.000
99%               1055.000                        107.972              327.000
max               4802.000                        365.000             2962.000
===== val
       gate_history_length  ga

## 5. Sanity checks

In [21]:
for name, g in gate.items():
    assert g["slate_id"].is_unique
    assert (g["gate_history_length"] >= 0).all() and (g["gate_purchase_count"] <= g["gate_history_length"]).all()
    assert g["gate_time_since_last_activity"].between(0, 366).all()
    assert not g.drop(columns=[]).isna().any().any()
print("basic checks OK")

# brute-force check on a small sample (val)
d = load(f"{DATA}/val.parquet", columns=HIST).head(200)
g = gate["val"].set_index("slate_id")
thr_age = np.float32(MIN_AGE_DAYS)
for sid, cands, wids, types, days in zip(d["slate_id"], d["candidate_wid_list"], d["history_wid_list"],
                                         d["history_type_list"], d["history_days_before"]):
    cs = set(int(c) for c in cands)
    k = [(t, np.float32(x)) for w, t, x in zip(wids, types, days)
         if not (np.float32(x) < thr_age and int(w) in cs)]
    assert g.loc[sid, "gate_history_length"] == len(k)
    assert g.loc[sid, "gate_purchase_count"] == sum(t == "ORD" for t, _ in k)
    if k:
        assert abs(g.loc[sid, "gate_time_since_last_activity"] - min(x for _, x in k)) < 1e-4
print("brute-force check on 200 val slates OK")

# train vs val vs test distribution (they should look alike)
print(pd.DataFrame({n: g[["gate_log_history_length", "gate_log_time_since_last", "gate_log_purchase_count"]].mean()
                    for n, g in gate.items()}).round(3))

basic checks OK
brute-force check on 200 val slates OK
                          train    val   test
gate_log_history_length   4.254  4.279  4.250
gate_log_time_since_last  0.803  0.824  0.835
gate_log_purchase_count   2.690  2.717  2.697


In [22]:
print(gate["train"][["gate_log_history_length","gate_log_time_since_last","gate_log_purchase_count"]].corr().round(2))

                          gate_log_history_length  gate_log_time_since_last  \
gate_log_history_length                      1.00                     -0.48   
gate_log_time_since_last                    -0.48                      1.00   
gate_log_purchase_count                      0.75                     -0.21   

                          gate_log_purchase_count  
gate_log_history_length                      0.75  
gate_log_time_since_last                    -0.21  
gate_log_purchase_count                      1.00  


## 6. Next steps
1. When Model 1 and Model 2 both produce scores, fit `a, b, c, bias` on **val** (full slates) to maximise ranking quality of `w·P1 + (1-w)·P2`, then report on test.
2. Before that, a useful evidence table: Model 1's NDCG@10 by gate bucket (history length, purchase count, time since last activity). If Model 1 is relatively better for short histories, the gate has a reason to exist.